## Processing RGB-encoded GeoTIFF masks from IRIS

After producing the AI-assisted cloud masks in IRIS, further processing is needed before use in the comparison. The IRIS reference masks have no georeferencing, meaning no spatial information is assigned. They are RGB-encoded, with "no cloud" pixels having a value of 255 and "cloud" pixels set as 0. This is the exact opposite of how the masks are labeled for all three algorithms being compared. Additionally, data type is int64. For these reasons, we need to process these GeoTIFF files prior to analysis.

### Imports

In [ ]:
import getpass
import time
import ee
import ee.batch
import glob
from pathlib import Path
import yaml  # For reading configuration files.

from google.colab import drive  # For mounting Google Drive in Colab.

import rasterio  # Read and write geospatial raster data
import numpy as np

# Must authenticate your EE account before use of the package.
project_id = getpass.getpass('Enter your EE Project ID: ')
ee.Authenticate()
ee.Initialize(project=project_id)

# This is how we can access our drive files in Colab.
drive.mount('/content/gdrive/My Drive')


In [ ]:
config_path = '/content/gdrive/My Drive/config.yaml'  # Path to your config.yaml file in Google Drive.

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

### Define region and dates for export

In [ ]:
# Define area of interest: Sandusky Bay, Ohio.
AOI = ee.Geometry.Polygon(config['polygon'])

# Define date ranges for image collection as list of tuples (start, end).
# Start and end dates should be in 'YYYY-MM-DD' format. End date is exclusive.   
date_ranges = [(dr['start'], dr['end']) for dr in config['date_ranges']]

### Export EE image to Google Drive

In [ ]:
# Use with export function to visualize the cloud mask in black and white.

def cloud_mask_vis(image):
    # Define the range of pixel values that will be mapped to 0-255.
    vis = image.visualize(**{
        'palette': ['black', 'white'],
        "min": 0,     # Pixel values at or below this will be displayed as black.
        "max": 0.4    # Pixel values at or above this will be displayed as white.
    })
    return vis

In [ ]:
# Export to a folder in Google Drive. If raw mask is used as input for
# the image parameter, the output will be binary (values strictly 0 or 1). Else,
# if the cloud_mask_vis function is used with the mask, the output will contain values from 0 to 255.

def export_image(mask, description, folder='Results', region=AOI, scale=10, crs='EPSG:4326', maxPixels=1e9):
    # Export cloud mask to Google Drive.
    task = ee.batch.Export.image.toDrive(
        image=mask,
        description=description,    # Filename.
        # The Google Drive Folder that the export will reside in. Note: 
        # (a) if the folder name exists at any level, the output is written to it, 
        # (b) if duplicate folder names exist, output is written to the most recently modified folder, 
        # (c) if the folder name does not exist, a new folder will be created at the root, and 
        # (d) folder names with separators (e.g. 'path/to/file') are interpreted as literal strings, 
        # not system paths. Defaults to Drive root.
        folder=folder,      
        region=region,
        scale=scale,    # Resolution in meters per pixel. Defaults to 1000.
        crs=crs,      # Default is global GCS; EPSG:32617 is specific to Ohio.
        maxPixels=maxPixels
    )
    task.start()

    print("Export started. Check your Earth Engine Tasks tab.")

    # Monitor the task status
    print(task.status()['id'])
    while task.status()['state'] != 'COMPLETED':
        print(task.status()['state'])
        time.sleep(60)

    print('Done')

### Write binary mask files to Drive (necessary for comparison)

In [ ]:
# Choose any cloud mask file from any of three algorithms to copy metadata.
# This is crucial to ensure that the reference masks are properly georeferenced.
sample_mask_path = Path(config['paths']['sample_mask_path'])

with rasterio.open(sample_mask_path) as ref:
    ref_meta = ref.meta.copy()

# Output cloud mask files from the IRIS program need to be uploaded to this directory
# in Google Drive before running the code below to convert them to georeferenced binary
# masks that align with the other two algorithms and can be used for evaluation.
iris_files = glob.glob(f'{Path(config["paths"]["reference_masks_root"])}/*.tif')

for iris_input_path in iris_files:
    iris_output_path = iris_input_path.replace(".tif", "_processed.tif")

    with rasterio.open(iris_input_path) as mask:
        # Mask information is in 3rd band (layer).
        iris_data = np.array(mask.read(3))
        # Convert 255 to 0 (no cloud), else 1 (cloud) and convert data type
        # to align with masks produced by the three algorithms.
        iris = np.where(iris_data == 255, 0, 1).astype(rasterio.uint8)

    # Open each mask file and write the newly processed
    # and georeferenced version to the output path.
    with rasterio.open(iris_output_path, "w", **ref_meta) as dest:
        # Write the NumPy array to the first band (band index starts at 1)
        dest.write(iris, 1)
        print("Unique values in saved mask:", np.unique(iris))
        print("CRS:", dest.crs)
        print("Transform:", dest.transform)


    print(f"Saved {iris_input_path} to: {iris_output_path}")

### Export RGB mask files to Drive (optional)

In [ ]:
# For this section to work, you must first download the processed IRIS mask files
# locally to your device, then manually upload them in the Assets tab of GEE.

# Specify the path to your folder or image collection.
asset_path = f'projects/{project_id}/assets/'

# List assets in the specified path.
asset_list = ee.data.listAssets(asset_path)

# Iterate through the list, print asset details, and export each as a mask file.
for asset in asset_list['assets']:
    iris_mask = ee.Image(asset['id'])

    # Rename the files to avoid overwriting when exporting
    asset_name = asset['id'].split('/')[-1]
    export_name = asset_name.replace('processed', 'mask')

    export_image(cloud_mask_vis(iris_mask), export_name, 'reference')

### Validation

In [ ]:
def check_mask(binary_mask):
    # Check unique values.
    unique_vals = np.unique(binary_mask)
    print(f"Unique values: {unique_vals}")
    print(f"Number of unique values: {len(unique_vals)}")

    # Confirm binary format (only 0s and 1s).
    is_binary = set(unique_vals).issubset({0, 1})
    print(f"Is binary (0,1): {is_binary}")

    # Check pixel values and percentage cloudy.
    print(f"Total pixels: {binary_mask.size}")
    print(f"Cloud pixels (value=1): {np.sum(binary_mask == 1)}")
    print(f"Clear pixels (value=0): {np.sum(binary_mask == 0)}")
    print(f"Percentage cloudy: {(np.sum(binary_mask == 1) / binary_mask.size) * 100:.2f}%")

    # Check data type.
    print(f"Data type: {binary_mask.dtype}")

In [ ]:
for ref_mask in glob.glob(f'{Path(config["paths"]["reference_masks_root"])}/*.tif'):
    with rasterio.open(ref_mask) as mask:
        data = np.array(mask.read())

    check_mask(data)